# Task 4.5 — Experiment on repository sizes

Run from the project root: `src/ifm_konwoledge_enginnering/Package_Manager/Project`

In [1]:
import time
from pathlib import Path
import pandas as pd

from package_manager.parsing import load_repository, parse_requests_text
from package_manager.install import install
from package_manager.verify import verify
from package_manager.kb import build_knowledge_base


In [2]:
def run_one(N):
    repo_path = BASE / f"repo_{N}.json"
    req_path = BASE / f"requests_{N}.txt"

    t0 = time.perf_counter()
    repo = load_repository(repo_path)
    variables, opt, axioms = build_knowledge_base(repo)
    build_time = time.perf_counter() - t0

    requests = parse_requests_text(req_path.read_text())

    t1 = time.perf_counter()
    try:
        installed = install(repo, requests)
        solve_time = time.perf_counter() - t1
        verified = verify(repo, installed, requests) == []
        return {
            "N": N,
            "versions": sum(len(p.versions) for p in repo.packages.values()),
            "axioms": len(axioms),
            "requests": requests,
            "installed": len(installed),
            "build_time": build_time,
            "solve_time": solve_time,
            "verified": verified,
        }
    except Exception as e:
        solve_time = time.perf_counter() - t1
        return {
            "N": N,
            "versions": sum(len(p.versions) for p in repo.packages.values()),
            "axioms": len(axioms),
            "requests": requests,
            "result": f"no solution: {e}",
            "build_time": build_time,
            "solve_time": solve_time,
        }


## Run Example

In [3]:
BASE = Path('examples')
result = run_one('example')

pd.DataFrame.from_dict(result, orient='index')

,0
N,example
versions,20
axioms,26
requests,"[app==2.0, tool]"
result,no solution: no solution for the given requests
build_time,0.013702
solve_time,0.010408


## Run  Experiment

In [4]:
GROUP = "k01"  # change to your group
BASE = Path("data/repos/repos") / GROUP
Ns = [10, 30, 100, 300, 1000]

results = [run_one(N) for N in Ns]

WARNING Package indigo-kit-lib=2.0.1 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='no-such-pkg', constraints=[])]) matches no version in the repository
WARNING Package quartz-kit=1.2 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='six', constraints=[Constraint(op='>=', version=<Version('999')>)])]) matches no version in the repository
WARNING Package quartz-kit=1.3 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='six', constraints=[Constraint(op='>=', version=<Version('2.0')>), Constraint(op='<', version=<Version('1.0')>)])]) matches no version in the repository
WARNING Package quartz-kit=2.0 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='quartz-native', constraints=[])]) matches no version in the repository
WARNING Package krypton-cli=1.2 requirement not satisfiable: requirement Requirement(alternatives=[Atom(package='pyrsistent', constraints=[Constrain

In [5]:
import pandas as pd
pd.DataFrame(results)


,N,versions,axioms,requests,installed,build_time,solve_time,verified
0,10,61,22,"[toml, six, xenon-api]",4,0.007927,0.029323,True
1,30,205,125,"[jsonschema-specifications, referencing, krypt...",7,0.040356,0.101254,True
2,100,1305,1346,"[typing, botocore, falcon-cli]",6,0.553218,0.963897,True
3,300,6659,6016,"[packaging, iniconfig-vortex, vortex-client]",33,3.180159,6.870159,True
4,1000,20866,16002,"[python-dateutil, typing-quill, xenon-api]",5,7.619638,14.057976,True
